# Assignment 5: Catching Data Leakage Before It Catches You
**Course:** Feature Engineering & MLOps  
**Name:** Sakshi Kore  
**Assignment No:** 5  
**Topic:** Target Leakage and Preprocessing Leakage  

In this assignment, we examine the mechanics of target leakage and preprocessing leakage using a telecom customer churn dataset, demonstrate how to detect both, and build an honest, deployable pipeline.

## 1. Imports and Setup

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, roc_auc_score

np.random.seed(42)
print("Libraries loaded.")

Libraries loaded.


## 4.1 — Target Leakage: Prove It, Don't Just Assert It

In [2]:
data_path = '../data/raw/customer_churn_a5.csv' if os.path.exists('../data/raw/customer_churn_a5.csv') else 'data/raw/customer_churn_a5.csv'
df = pd.read_csv(data_path)

# Single line check confirming non-null cancellations equals churn=1
matches_exact = (df['days_since_cancellation'].notnull().sum()) == (df['churn'] == 1).sum()
print(f"Non-null days_since_cancellation count: {df['days_since_cancellation'].notnull().sum()}")
print(f"Number of churn=1 rows:                {(df['churn'] == 1).sum()}")
print(f"Confirmed equal in a single line of code: {matches_exact}")

Non-null days_since_cancellation count: 149
Number of churn=1 rows:                149
Confirmed equal in a single line of code: True


### Correlation Comparison

In [3]:
df_check = df.copy()
df_check['final_bill_amount'] = df_check['final_bill_amount'].fillna(-1)

corr_final_bill = df_check['final_bill_amount'].corr(df['churn'])
corr_tenure = df_check['tenure_months'].corr(df['churn'])

print(f"Correlation with churn:")
print(f"  final_bill_amount (leaked): {corr_final_bill:.4f}")
print(f"  tenure_months (legitimate): {corr_tenure:.4f}")

Correlation with churn:
  final_bill_amount (leaked): 0.9250
  tenure_months (legitimate): -0.2207


**Observation:** `final_bill_amount` shows a correlation of 0.9250 with churn, which is suspiciously high compared to the legitimate feature `tenure_months` (-0.2207). This indicates that the feature is an outcome of churn rather than an antecedent cause.

### Training Model A (Legitimate) vs. Model B (Leaked)

In [4]:
# One-hot encode contract_type
df_encoded = pd.get_dummies(df, columns=['contract_type'], drop_first=True)

legit_features = ['tenure_months', 'monthly_charges', 'support_calls'] + [c for c in df_encoded.columns if c.startswith('contract_type_')]
leaked_features = legit_features + ['days_since_cancellation', 'final_bill_amount']

# Split 75/25
X_train, X_test, y_train, y_test = train_test_split(df_encoded, df_encoded['churn'], test_size=0.25, random_state=42)

# Train Model A (honest features)
scaler_A = StandardScaler()
X_tr_A = scaler_A.fit_transform(X_train[legit_features])
X_te_A = scaler_A.transform(X_test[legit_features])

model_A = LogisticRegression(random_state=42).fit(X_tr_A, y_train)
acc_A = accuracy_score(y_test, model_A.predict(X_te_A))
auc_A = roc_auc_score(y_test, model_A.predict_proba(X_te_A)[:, 1])

# Train Model B (leaked features)
X_tr_B_raw = X_train[leaked_features].fillna(-1)
X_te_B_raw = X_test[leaked_features].fillna(-1)

scaler_B = StandardScaler()
X_tr_B = scaler_B.fit_transform(X_tr_B_raw)
X_te_B = scaler_B.transform(X_te_B_raw)

model_B = LogisticRegression(random_state=42).fit(X_tr_B, y_train)
acc_B = accuracy_score(y_test, model_B.predict(X_te_B))
auc_B = roc_auc_score(y_test, model_B.predict_proba(X_te_B)[:, 1])

print(f"Model A (Honest Features): Accuracy = {acc_A:.4f}, ROC-AUC = {auc_A:.4f}")
print(f"Model B (Leaked Features): Accuracy = {acc_B:.4f}, ROC-AUC = {auc_B:.4f}")

Model A (Honest Features): Accuracy = 0.8000, ROC-AUC = 0.8086
Model B (Leaked Features): Accuracy = 1.0000, ROC-AUC = 1.0000


**Performance Gap and Production Consequence:**  
Model B achieves an unrealistic 100% accuracy and 1.0000 ROC-AUC because it uses post-churn variables. If deployed in production, `days_since_cancellation` and `final_bill_amount` would not exist for active customers because they have not cancelled yet. The model would receive null values for its primary predictive drivers and its real-world performance would drop to near-useless levels.

## 4.2 — Preprocessing Leakage: The Wrong Order vs. The Right Order

In [5]:
numeric_cols = ['tenure_months', 'monthly_charges', 'support_calls']

# 1. Wrong order: scaling before split
scaler_wrong = StandardScaler().fit(df[numeric_cols])
print("Wrong Scaler (Full Data):")
print("  Learned Mean: ", np.round(scaler_wrong.mean_, 4))
print("  Learned Scale:", np.round(scaler_wrong.scale_, 4))

# 2. Right order: scaling after split
scaler_right = StandardScaler().fit(X_train[numeric_cols])
print("\nCorrect Scaler (Train Only):")
print("  Learned Mean: ", np.round(scaler_right.mean_, 4))
print("  Learned Scale:", np.round(scaler_right.scale_, 4))

diff_means = scaler_wrong.mean_ - scaler_right.mean_
print(f"\nDifference in learned means: {np.round(diff_means, 4)}")

Wrong Scaler (Full Data):
  Learned Mean:  [19.5617 65.922   2.165 ]
  Learned Scale: [17.5245 22.7993  1.4981]

Correct Scaler (Train Only):
  Learned Mean:  [19.6933 65.3928  2.1778]
  Learned Scale: [17.4944 22.629   1.5024]

Difference in learned means: [-0.1317  0.5292 -0.0128]


**Why the Principle Matters:** Even though the difference in means is modest here, fitting a scaler across the full dataset allows information about the test set's distribution to contaminate training. In real production scenarios with temporal shifts or small datasets, this leaks distributional knowledge and yields overly optimistic validation scores.

### 4. Rebuilding with Scikit-Learn Pipeline

In [6]:
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(random_state=42))
])

# Fit strictly on train split
pipe.fit(X_train[legit_features], y_train)

pipe_scaler = pipe.named_steps['scaler']
print("Pipeline Scaler Means (first 3 numeric):", np.round(pipe_scaler.mean_[:3], 4))
print("Matches Manual Correct Scaler:          ", np.allclose(pipe_scaler.mean_[:3], scaler_right.mean_))

Pipeline Scaler Means (first 3 numeric): [19.6933 65.3928  2.1778]
Matches Manual Correct Scaler:           True


## 4.3 — The Fix: Honest, Deployable Model

In [7]:
pipe_preds = pipe.predict(X_test[legit_features])
pipe_probs = pipe.predict_proba(X_test[legit_features])[:, 1]

final_acc = accuracy_score(y_test, pipe_preds)
final_auc = roc_auc_score(y_test, pipe_probs)

print(f"Final Honest Pipeline Test Accuracy: {final_acc:.4f}")
print(f"Final Honest Pipeline Test ROC-AUC:  {final_auc:.4f}")

assert np.isclose(final_acc, acc_A) and np.isclose(final_auc, auc_A), "Discrepancy with Model A!"
print("Verified: Pipeline performance exactly matches honest Model A.")

Final Honest Pipeline Test Accuracy: 0.8000
Final Honest Pipeline Test ROC-AUC:  0.8086
Verified: Pipeline performance exactly matches honest Model A.


## Bonus (+10% Extra Credit): Cross-Validation Leakage

In [8]:
# 1. Leaked CV: scaling whole dataset prior to cross-validation
scaler_global = StandardScaler()
X_all_scaled = scaler_global.fit_transform(df_encoded[legit_features])
cv_leaked = cross_val_score(LogisticRegression(random_state=42), X_all_scaled, df_encoded['churn'], cv=5)

# 2. Clean CV: scaler encapsulated inside the pipeline
cv_clean = cross_val_score(pipe, df_encoded[legit_features], df_encoded['churn'], cv=5)

print(f"Leaked CV Accuracy: {cv_leaked.mean():.4f} (+/- {cv_leaked.std():.4f})")
print(f"Clean CV Accuracy:  {cv_clean.mean():.4f} (+/- {cv_clean.std():.4f})")

Leaked CV Accuracy: 0.7767 (+/- 0.0193)
Clean CV Accuracy:  0.7767 (+/- 0.0193)


**Explanation:** On this dataset, the cross-validation mean scores are virtually identical (0.7767) because the sample size is moderate (600 rows) and the numeric distributions are fairly stable across folds. However, when datasets have extreme outliers or when feature selection is performed prior to cross-validation, the leakage can cause significant optimistic bias.

# Question Answers

Here are the direct, compiled answers to all questions asked in this assignment and related course notes:

**Q1: How many rows have a non-null `days_since_cancellation`, and does it exactly equal the number of `churn=1` rows?**  
**A:** Exactly 149 rows, which matches the 149 `churn=1` rows 100% (`(df['days_since_cancellation'].notnull().sum()) == (df['churn'] == 1).sum()` evaluated to `True`).

**Q2: Is the correlation of `final_bill_amount` against churn suspiciously high? How does it compare to `tenure_months`?**  
**A:** Yes, it is suspiciously high at **0.9250**. In contrast, a legitimate feature like `tenure_months` has a correlation of **-0.2207**. A correlation above 0.90 is a classic sign of target leakage.

**Q3: What is the accuracy and ROC-AUC gap between Model A (honest) and Model B (leaked)?**  
**A:** Model A achieves 80.00% accuracy and 0.8086 ROC-AUC, while Model B achieves an artificial 100.00% accuracy and 1.0000 ROC-AUC (a 20.00% accuracy gap).

**Q4: Exactly what would happen if Model B were deployed to score brand-new applicants who haven't decided anything yet?**  
**A:** For new or active customers, `days_since_cancellation` and `final_bill_amount` do not exist because they have not cancelled yet. The model would receive null/missing values for its two most important features, causing its real-world performance to collapse.

**Q5: What is the "could this exist at prediction time?" test?**  
**A:** It is a diagnostic test where you verify whether a feature is genuinely known and recorded at the exact moment the model makes a prediction. If the feature only exists after or because the outcome occurs, it is target leakage.

**Q6: What is the numeric difference between the wrong scaler (full dataset) and the correct scaler (train only)?**  
**A:** The mean difference is `[-0.1317, 0.5292, -0.0128]` for `tenure_months`, `monthly_charges`, and `support_calls`.

**Q7: Why does the preprocessing leakage principle matter regardless of how small the gap is on this dataset?**  
**A:** Fitting scalers on test data leaks future distribution parameters into training. While the gap is small here due to a balanced sample, in real-world pipelines with smaller datasets, data drift, or extreme outliers, it leads to severe optimistic evaluation bias.

**Q8: Why is an `sklearn.pipeline.Pipeline` useful for preventing leakage?**  
**A:** A Pipeline bundles transformers and the estimator into a single object, ensuring transformers are only ever fitted on the training split during `.fit()`, and then applied without refitting during `.predict()`.

**Q9: What is the difference between target leakage and look-ahead bias?**  
**A:** Target leakage occurs when a feature is a direct result or proxy of the target itself. Look-ahead bias occurs when a feature uses data from a timestamp after the prediction point, even if not derived from the target directly.

**Q10: Why can cross-validation leak if preprocessing is fit before the CV split, and why was the difference small here?**  
**A:** Fitting a scaler on the whole dataset before CV lets test fold information leak into the training folds. The score difference was small here (0.7767 for both) because the dataset is well-behaved with 600 rows and stable distributions across folds, but in high-dimensional or small datasets, it can create significant bias.